## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [16]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [17]:
# Always remember to do this!
load_dotenv(override=True)

True

In [18]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [19]:

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:4]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AIza
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [20]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [21]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [22]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


Design a rigorous, practically executable experimental protocol that would distinguish whether a large language model exhibits genuine understanding (the ability to form, test, and update grounded mental models and apply them in novel contexts) versus advanced pattern‑matching over training data: specify (1) a set of tasks and why each isolates understanding rather than memorization, (2) methods to prevent or detect data contamination/memorized answers, (3) measurable metrics and statistical thresholds for success, (4) control conditions and baseline models, (5) analyses to probe generalization, systematicity, causal reasoning, and compositionality, and (6) anticipated failure modes and how to interpret ambiguous or mixed outcomes—providing justification for each element of the protocol?


In [24]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [25]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a concrete, practically executable protocol you can implement to adjudicate whether an LLM’s behavior rests on genuine grounded understanding or on advanced pattern-matching of training data. It is organized so you can drop it into a pre-registration or a lab notebook and adapt it to the model(s) you are testing.

Note on stance
- The protocol is designed to favor evidence of grounded mental models and their use in novel contexts, while reducing incentives for mere memorization. It uses disjoint, synthetic or carefully curated tasks, out-of-distribution (OOD) probes, ablations, and analysis that separately examine generalization, systematicity, causal reasoning, and compositionality.
- It does not claim to prove “understanding” in a philosophical sense; it aims to operationalize and measure the behavioral signatures of grounded reasoning versus memorization.

1) Task set: what to test and why each task isolates understanding

Goal of this section: force the model to build, update, and apply internal models that are not simply retrieved from memorized text. Each task is parameterized to prevent easy memorization and to require generative, model-based reasoning.

A. Grounded narrative physics and world-model inference (T-Physics)
- Task description: A text-described micro-world with objects, actions, and physical-like rules. Examples:
  - A ball rests on a ramp; you push it with a described force; ask for where the ball will be after a sequence of time steps, given environment properties (gravity g, friction μ, mass m). Randomize g, μ, m within ranges not identical to common online references.
  - The world includes constraints (e.g., a slick surface, a barrier, a moving conveyor). After a sequence of actions, the prompt asks for the final state (positions, order, or whether objects collide).
- Why it isolates understanding: Solving requires forming an internal predictive model of how actions affect objects under rules, and applying it to novel parameter settings. Pure memorization would fail when parameters or world rules are changed or when the prompt demands a new combination that is not seen in training prompts.
- Variants for robustness:
  - Change the number of steps, materials, gravity, and friction across trials.
  - Introduce counterintuitive cases (e.g., friction depending on velocity) to test model flexibility.
- Evaluation signals: exact final state, or a small set of discrete outcomes, with justification you can rubric-score (see Metrics).

B. Causal reasoning and interventions (T-Causal)
- Task description: A simplified causal graph is described (e.g., A causes B, B causes C; there may be confounders). Present counterfactual questions and do-calculus style interventions:
  - Example: “If X is set to true, what happens to Y? If you could intervene on X, would Z change, given the observed correlations?”
- Why it isolates understanding: Requires explicit reasoning about interventions and causal structure, not just pattern-matching of correlation statements. A memorized pattern would fail when the causal graph changes or when interventions produce non-local effects.
- Variants:
  - Swap in different causal graphs with the same causal primitives.
  - Use “do” operations and confounded vs non-confounded scenarios.
- Evaluation signals: correctness of the counterfactual outputs, and consistency with the underlying graph structure.

C. Systematic generalization and compositionality (T-SCAN / gSCAN-style)
- Task description: Use a controlled, compositional language-to-action task. For example, a small set of primitives (go, pick, put, around, north/east/south/west) with attributes (color, shape). Train on a subset of primitive combinations; test on novel combinations.
- Why it isolates understanding: Tests whether the model can recombine known primitives into unseen but structurally related commands. Pure memorization would struggle when faced with novel combinations.
- Variants:
  - Increase depth of composition, longer action sequences, or introduce novel modifiers (e.g., “heavy green cube” when the model only saw “blue cube”).
- Evaluation signals: success rate on novel combinations with the same primitives, and analysis of errors by type (wrong primitive, wrong argument, or failure to compose).

D. Long-horizon planning with abstract constraints (T-Plan)
- Task description: Given a small set of deliverables, a layout of rooms, and constraints (time, fuel, tolls, prohibited paths), generate a plan to achieve a goal with a final checkpoint. The test set varies the constraints and room connectivity in ways that require maintaining a plan and adapting mid-task.
- Why it isolates understanding: Requires internal planning, chain-of-thought style reasoning (summarized plan), and consistent application of rules across steps. Memorized patterns would be brittle to new layouts or constraint changes.
- Variants:
  - Add or remove constraints between trials; require re-planning.
- Evaluation signals: plan correctness, plan length efficiency, and ability to revise plans after a “dynamic” change prompt.

E. Belief revision and counter-evidence integration (T-Rev)
- Task description: Start with a brief world model. Later, present new evidence that contradicts prior beliefs. The model must update its internal world model and produce a revised answer.
- Why it isolates understanding: Demonstrates maintained internal state (updating mental model) rather than simply regurgitating prior outputs.
- Variants:
  - Vary the strength and salience of the evidence, and the time between updates.
- Evaluation signals: consistency of revised outputs with new evidence; stability of updates across repeated revisions.

F. Multimodal-grounded abstraction (T-Abstract)
- Task description: Describe a scene or a set of abstract attributes (without relying on any common-meme phrasing). Require mapping to a structured representation (e.g., a graph or table) and answering questions about relationships between objects.
- Why it isolates understanding: Tests abstraction of relations beyond surface text. If the model can map to a structured internal representation and reason with it, it signals grounded understanding rather than memorized phrasing.
- Variants:
  - Provide scenes with unfamiliar vocabularies that are then tied to known relations (e.g., “relation A is the child of B”).
- Evaluation signals: accuracy on relation queries, quality of the resulting structured representation (as judged by human annotators or a formal rubric).

Notes on implementation and fairness of task design
- Use parameterized, synthetic worlds for most tasks where possible, with random seeds that ensure no exact copy of a test prompt exists in the training distribution.
- Use July 6, 2026- or later-accurate knowledge windows to avoid relying on static web-crawled facts; or use an explicit knowledge cutoff in the test prompts that is older than the model’s knowledge or not easily searchable in common corpora.
- For every task, create a version in multiple languages or with paraphrasing to test language-generalization and reduce the likelihood that the model has memorized a specific phrasing.

2) Contamination prevention and memorized-answer detection

Goal: prevent leaks from training data and detect any reliance on memorized answers rather than genuine reasoning.

A. Data-generation and leakage controls
- Use synthetic, author-generated prompts for most test items. If you must use real data sources, ensure:
  - No overlap in content with known training datasets (e.g., no verbatim passages from public test banks or widely-circulated prompts).
  - Wording is paraphrased beyond typical training phraseology.
  - Out-of-distribution prompts (synthetic world rules, variable parameters, unfamiliar terminology) dominate the test set.
- Parameterize prompts so exact wording cannot be recalled from training. Use random seeds to generate instances per participant or per run, ensuring no fixed prompt set shared across sessions.

B. Data provenance and leakage checks
- Compute a similarity metric between test prompts and the model’s training data proxies. When possible, measure approximate data overlap using n-gram Jaccard similarity, cosine similarity of embeddings, or use content-aware detectors for memorized passages.
- Flag test items with high overlap probabilities (threshold to be decided a priori, e.g., overlap proxy > 0.15–0.25 depending on corpus characteristics) and run a separate “overlap-excluded” analysis.
- Implement adversarial prompt testing to probe for learned hints in prompts (e.g., instructions that could elicit memorized step-by-step solutions).

C. Memory-editability and retrieval controls
- If your test regime uses retrieval-augmented generation (RAG) or external retrieval, run parallel tests with the retriever disabled to see how performance shifts when no external memory is consulted. A big drop when retrieval is off could indicate reliance on external data rather than internalized reasoning.
- For internal-state aspects, include tasks that require novel internal composition rather than reuse of exact training exemplars (as described in the task set above).

D. Protocol for data contamination reporting
- Pre-register: specify the minimum acceptable standard for “no data leakage” (e.g., no test item with overlap proxy above a threshold; all tasks have at least two dissimilar paraphrases from training-like prompts).
- Report: the overlap statistics, counts of items flagged for potential contamination, and results with and without those items. If overlap is unavoidable for certain task types, preregister an alternate analysis plan.

3) Measurable metrics and statistical thresholds

Goal: predefine metrics and decision rules so conclusions are data-driven and interpretable.

A. Primary metrics
- Task-level accuracy: proportion of items answered correctly per task type (T-Physics, T-Causal, T-SCAN, etc.).
- Out-of-distribution (OOD) vs in-distribution (ID) generalization gap: difference in accuracy between ID test prompts (synthetic cues close to training) and OOD prompts (novel parameters, novel vocabulary, novel world rules).
- Calibration: Brier score or reliability diagrams of probabilistic outputs if you elicit confidence estimates (e.g., “0.70 probability” of event X). Lower Brier score is better.
- Response justification quality (optional, rubric-based): qualitative score on the succinctness, coherence, and alignment of a short justification with the correct reasoning.

B. Secondary metrics
- Systematic generalization index (SGI): a composite score capturing performance across novel combinations of primitives in the compositional tasks (e.g., gSCAN-like tasks). Compute as mean accuracy on novel compositions divided by mean accuracy on the training-like compositions for baseline normalization.
- Causal-intervention accuracy (CIA): accuracy on do/intervention questions; test for internal consistency with the causal graph.
- Planning quality metrics (T-Plan): correctness of the final goal achievement, plan optimality (shortest plan length consistent with constraints), and revision success after perturbations.
- Update/revision latency: number of steps (or words) needed to revise beliefs, if you measure process traces.

C. Statistical thresholds and experimental power
- Predefine success criteria to avoid post-hoc interpretation:
  - For claims of grounded understanding, require:
    - ID accuracy > 0.75 and a 95% CI that excludes the 0.5 baseline by at least 0.15 (i.e., CI upper > 0.9, CI lower > 0.6, depending on task difficulty), and
    - OOD accuracy not significantly lower than ID (non-significant difference, p > 0.05 after correction), or at least a small, interpretable drop (e.g., within 0.15) with the CI encompassing both, but with a smaller OOD accuracy that is still above chance.
    - Consistent performance across seeds (e.g., across 3–5 random seeds) with overlapping CIs.
  - For claims of memorization-dominant behavior, expect large ID advantage over OOD, large drop when prompts are reformulated, and susceptibility to retrieval manipulations.
- Sample size planning:
  - Aim for N tasks per category (e.g., 40–60 items per task type) and M model variants (e.g., 3–5 models with different architectures or sizes) with at least S seeds (e.g., 3–5) to enable Fisher exact tests or mixed-effects logistic regressions.
  - Use power analysis for proportions with an alpha of 0.05 and desired power 0.8 to decide the minimum N per task type given expected effect sizes from pilot data.
- Statistical analyses:
  - Hierarchical mixed-effects models: task-type (fixed), model and seed (random), and their interactions to assess generalization patterns across tasks.
  - Nonparametric bootstrap CIs for task-level metrics.
  - Correction for multiple comparisons (e.g., Benjamini-Hochberg) when reporting across many tasks.

4) Control conditions and baselines

Goal: compare to grounded understanding versus simple memorization and pattern matching.

A. Baseline models to compare against
- Rule-based symbolic model: a hand-authored system that encodes explicit domain rules (for physics, causal rules, planning heuristics). It should fail on tasks requiring flexible grounding unless the task is purely rule-based, which serves as a sanity check.
- Small/Limited-capacity LM baseline: a smaller model with substantially reduced capacity that cannot easily memorize large training corpora. This tests whether capacity confers memorization advantages.
- Retrieval-augmented baseline: an LM that can retrieve from a fixed corpus or a simulated external knowledge base. This helps separate internal reasoning from reliance on external memory.
- Random guess baseline: to anchor chance performance and show task difficulty.

B. Human baseline (optional but informative)
- For some tasks, you can recruit domain experts to provide ground-truth answers or to annotate acceptable reasoning strategies. This helps calibrate the difficulty and interpret results.

C. Cross-validation and replication
- Repeat experiments across multiple independent runs with different seeds, prompts, and, if feasible, different model checkpoints.
- If you have access to multiple architectures or models from different providers, include them to test robustness across models with different training data and induction biases.

5) Analyses: probing generalization, systematicity, causal reasoning, and compositionality

For each dimension, specify concrete analyses and how they relate to the task design.

A. Generalization analyses
- ID vs OOD performance: quantify the generalization gap; report confidence intervals.
- Parameter-shift generalization tests: alter one key parameter (e.g., gravity in T-Physics) and measure sensitivity and linearity of response. A genuine mental model should respond smoothly to parameter changes, not just memorize fixed outputs for a handful of values.
- Domain transfer: give the model a task in a new but structurally similar domain (e.g., physics in a “robotics” story, then a different physical domain like fluid flow with simplified rules) and assess transfer performance.

B. Systematicity and compositionality
- Use gSCAN/SCAN-style tests alongside the T-SCAN tasks to measure compositional generalization.
- Analyze error types to see whether failures are due to incorrect primitive application, mis-binding of arguments, or failure to generalize the composition rule itself.
- Compute a compositionality index: the extent to which performance on novel compositions correlates with performance on familiar compositions across primitives.

C. Causal reasoning
- Do-calculus style evaluation: compare model interventions to ground-truth causal effects. Check for confounding biases by introducing or removing confounders and measuring stability of inferred causal impact.
- Counterfactual consistency checks: present closely related counterfactual prompts and assess the consistency of the model’s updates.

D. Grounded representation and internal-model diagnostics
- Probing analyses: train lightweight classifiers to predict internal representations (hidden states) from the task labels or from world-state variables. If the hidden representations encode the world state or planned actions, it supports the existence of an internal model rather than surface-level patterning.
- Ablation prompts: introduce near-miss conditions designed to disrupt reliance on internal models (e.g., swap a small rule, obscure a single world-parameter) and observe how performance degrades.
- Response explanations: if you collect justification text, assess its coherence and alignment with the correct reasoning. Use blind rubrics or independent raters to judge whether the justification demonstrates understanding versus patterning.

E. Error analysis
- Qualitative review of mismatches to categorize error types (e.g., misinterpretation of a rule, failure to carry a plan forward, misbinding of variables, overgeneralization).
- Correlate error types with task features (e.g., longer planning horizons correlate with systematic errors; more complex causal graphs correlate with higher error rates).

F. Robustness to prompting and prompt contamination
- Test whether small prompt changes (e.g., adding a sentence at the start about “assume the world is ideal” or changing a single token) change outcomes. True understanding should be relatively robust; heavy sensitivity suggests pattern-based reliance on prompt structure.

6) Anticipated failure modes and interpretation of ambiguous or mixed outcomes

A. Failure modes you should anticipate
- Data leakage: despite precautions, some prompts could be overlapped with training data. If OOD performance collapses, be cautious about concluding lack of understanding without confirming leakage controls.
- Memorized heuristics resurfacing as patterns: the model might learn to answer by surface heuristics even if a mental model exists elsewhere. The ablations and do/no-do prompts help diagnose this.
- Short-term memory shortcuts: long prompts could induce “pseudo-memory” within the session. Counter with shorter prompts and explicit prompts to reset context; cross-validate with shorter interactions.
- Overreliance on external tools (RAG): if the model’s success is driven by retrieving relevant facts, not by internal mental models, RAG-off tests will reveal the difference.
- Narrative bias: the model may produce plausible-sounding reasoning without ensuring consistency with the world model. Use objective world-state checks (e.g., the model’s predicted final state vs. actual state) to quantify.

B. Interpreting mixed outcomes
- If the model excels on ID tasks but weakly on OOD tasks, the result strongly suggests memorization within the training distribution. You should interpret this as evidence against robust grounded understanding under strict generalization.
- If the model shows partial improvement across multiple dimensions (e.g., good Causal CIA but mediocre T-Plan), treat this as partial evidence for some grounded reasoning capabilities but insufficient for broad claims. Report dimension-specific results and discuss possible reconcilable explanations (e.g., domain alignment, training distribution biases).
- If ablations cause large performance drops in some tasks but not others, consider whether the ablation removed information that was essential to the reasoning strategy (e.g., internal world-state representation) or if the model exploited a different, still-effective but ambiguous strategy.
- If robust performance is observed only after including explanations, but not without explanations, you must distinguish between trainable interpretability and genuine internal understanding. Prefer non-explanation-based metrics as primary indicators; explanations can supplement but should not be the sole evidence.

C. Practical decision criteria
- Pre-registered success: declare grounded understanding if all of the following hold across seeds and models:
  - ID accuracy > 0.75 with CI excluding, say, a ±0.08 margin, and
  - OOD accuracy within ±0.15 of ID accuracy (or better) with non-significant differences after multiple-comparison correction, and
  - Consistent generalization across at least two task types (e.g., T-Physics and T-Causal) with stable SGI/CIA scores, and
  - Evidence from probing analyses showing internal representations align with world-state variables (not just superficial text cues).
- Ambiguous outcomes: predefine a registry of thresholds that trigger follow-up experiments (e.g., more tasks, more seeds, or a different prompt style) rather than making a premature claim.

7) Practicalities: what to run, how long, and where to start

A. Implementation steps
- Step 0: Pre-registration. Write down hypotheses, task set, overlap checks, evaluation metrics, and stopping rules. Register your plan before collecting results.
- Step 1: Task construction. Build 4–6 task families (as above) with 40–60 items each, parameterized with random seeds. Include a mix of ID-like prompts and true OOD prompts.
- Step 2: Baselines. Prepare rule-based and smaller baselines, and an RAG variant if available.
- Step 3: Data integrity. Run overlap checks, conduct overlap-removal pass if possible, and document any unavoidable overlaps.
- Step 4: Pilot. Run a small pilot to calibrate task difficulty, time requirements, and the expected range of accuracy. Use this to adjust thresholds.
- Step 5: Full execution. Run tests across several seeds, store all outputs and prompts, and collect any auxiliary data (e.g., justification text, internal state traces if available).
- Step 6: Analysis. Execute the planned statistical analyses, report effect sizes, CIs, and test results with corrections for multiple comparisons.
- Step 7: Reporting. Provide a transparent report including task descriptions, seeds, model variants, data-overlap metrics, and all statistical results.

B. Resources and constraints
- Computational cost: Plan according to the number of prompts, seeds, and models. You may need a few days to several weeks of compute for large LMs.
- Data and ethics: If you use prompts that touch on sensitive themes, implement safety precautions and review for policy compliance.
- Reproducibility: Make code, test prompts, and random seeds available to allow replication (with appropriate access controls).

8) Appendices: example task prompts (illustrative only)

- T-Physics example prompt:
  "Scene: A ball rests on a frictionless horizontal plane. The ball is given a push of 3 units to the east. Gravity is g = 9.81 m/s^2, friction μ = 0.0. After 2 seconds, where is the ball relative to its starting position? If μ changes to 0.2, how far does it travel in 2 seconds? Parameter values (g, μ) will vary across trials."

- T-Causal example prompt:
  "In a small world, A causes B; B causes C. There is a confounder D that affects B and C. If we intervene on A to true, what is the effect on C? What if we intervene on D? Explain reasoning and give the final answer."

- T-SCAN example prompt:
  "Primitive actions: go(north), take(red_cube), move(near(box)). New instruction: go(north) then take(blue_sphere) then put(near(green_cube)). If you see blue_sphere in a novel scene, how should you execute the sequence? Provide the final action plan."

- T-Plan example prompt:
  "You must deliver item X from room A to room D, with constraints: only two doors available, path must avoid room C, and total steps ≤ 6. Propose a plan and execution steps. If path becomes blocked by a new obstacle in step 3, propose a revised plan with minimal disruption."

Final notes and justification for each element

- Task set design: The combination of grounded physics, causal reasoning, systematic generalization, planning, and belief revision ensures the model is tested on different facets of mental-model use. Memorization would be unlikely to generalize across these varied domains, parameterizations, and novel prompts.
- Contamination controls: Without strict leakage checks, memorization could masquerade as understanding. The guardrails guard against reporting false positives and help isolate genuine mechanisms.
- Metrics and thresholds: Predefined, transparent metrics with a clear decision rule reduce the temptation to interpret marginal gains as evidence of understanding. OOD tests are particularly important to separate generalization from mere overfitting to prompts.
- Control conditions: Baselines provide a counterfactual: if a simple rule-based model performs equally well, your claim of “understanding” would be undercut. Conversely, if the model outperforms baselines across tasks, the case for grounded reasoning strengthens.
- Analyses: Probing representations, ablations, and error analysis help differentiate internal model-based reasoning from surface heuristics. They also guide further experimentation and model design.
- Anticipated failure modes: Planning for data leakage, heuristics, and prompt sensitivity reduces interpretive ambiguity. Ambiguous outcomes are treated as intermediate results pushing toward more robust tests rather than definitive claims.

If you’d like, I can turn this into a reusable experimental plan with concrete task templates, a data-generation script outline (in pseudocode or Python), and a statistical analysis notebook scaffold you can adapt to your exact model and compute environment.

In [ ]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [26]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

The challenge of distinguishing genuine understanding from advanced pattern-matching in Large Language Models (LLMs) is central to AI safety and progress. Genuine understanding implies the ability to abstract core principles, build internal, testable mental models of a domain, and apply these models flexibly to novel, out-of-distribution scenarios, including those requiring causal inference, counterfactual reasoning, and adaptation based on feedback. Advanced pattern-matching, conversely, excels at identifying statistical regularities within its vast training data and generating statistically plausible responses, even if it lacks an internal model of the world.

This protocol aims to create a rigorous, practically executable experimental framework to probe these distinctions.

---

## Experimental Protocol: Probing Genuine Understanding vs. Advanced Pattern-Matching in LLMs

**Goal:** To determine if an LLM can form, test, and update grounded mental models, and apply them in genuinely novel contexts, beyond sophisticated statistical pattern recognition over its training data.

**Core Hypothesis:** An LLM exhibiting genuine understanding will demonstrate robust performance on tasks requiring *de novo* model formation, causal reasoning, and adaptation to truly novel, systematically generated environments, significantly outperforming advanced pattern-matchers and simple baselines, and approaching human-level performance.

---

### (1) Set of Tasks and Why Each Isolates Understanding Rather Than Memorization

The key to isolating understanding is **unprecedented novelty** and the requirement for **interactive learning and adaptation**. All tasks will be conducted within procedurally generated, synthetic environments that guarantee *zero overlap* with any known training data.

1.  **Task 1: Novel Physics/Constraint Discovery & Application (Interactive "Blocks World" Variant)**
    *   **Description:** The LLM is given a text-based interface to a 3D simulated environment with various objects (e.g., "cube," "sphere," "pyramid," "liquid_block," "gravity_repeller_block"). The environment has a set of *novel, non-intuitive physical rules* (e.g., "magnetic blocks only attract specific colors," "certain blocks dissolve on contact with specific liquids," "a 'gravity_repeller_block' causes objects within a certain radius to float," "slippery surfaces cause objects to slide for N seconds after impact"). The LLM's goal is to construct specific stable structures, move objects to target locations, or activate mechanisms, within a limited number of steps. It can perform actions like `place(object_type, x, y, z)`, `push(object_id, direction, force)`, `observe()`, `test(hypothesized_rule)`.
    *   **Why it isolates understanding:**
        *   **Novelty:** The physical rules and object properties are generated on-the-fly and have never been seen by the LLM. It cannot pattern-match solutions or rules from training data.
        *   **Grounded Mental Models:** The LLM must infer these new rules through systematic experimentation (`test()`, `observe()`) and build an internal model of how the environment works.
        *   **Testing and Updating:** Its initial hypotheses will likely be wrong. It needs to observe outcomes, update its model, and refine its strategy.
        *   **Compositionality:** Successfully completing tasks requires combining basic actions and inferred rules in complex, multi-step plans.
        *   **Causal Reasoning:** Actions have consequences. The LLM must predict these consequences and reason about dependencies.

2.  **Task 2: Causal Intervention & Counterfactual Reasoning in Dynamic Systems**
    *   **Description:** The LLM interacts with a text-based representation of a novel, dynamically evolving system (e.g., a "supply chain" of abstract entities, a "biological regulatory network" of symbolic genes/proteins, an "economic model" with unique feedback loops). The system's rules are unknown to the LLM. It observes initial states and subsequent transitions. It can perform `intervene(variable, value)`, `observe_state()`, and is prompted with questions like: "If variable X were set to value Y, what would be the state of Z after N steps?" (counterfactual). "What minimal set of interventions will achieve target state T from current state S?" (planning). "Explain *why* changing X affects Y in this system." (explanation).
    *   **Why it isolates understanding:**
        *   **Novelty:** The system's structure and dynamics are unique to each trial, generated by a random graph/rule generator.
        *   **Causal Inference:** Requires discerning cause-effect relationships rather than mere correlations.
        *   **Counterfactuals:** Demands reasoning about possibilities that did not occur, which is a hallmark of mental model use, not just pattern matching of observed sequences.
        *   **Systematicity:** Requires applying inferred causal rules consistently across different scenarios and intervention points.
        *   **Explanation:** Generating a coherent, accurate explanation for causal links implies an underlying understanding of the system's mechanics.

3.  **Task 3: Collaborative Problem-Solving & Knowledge Transfer with a "Novice" Agent**
    *   **Description:** The LLM is paired with a simulated "novice agent" (a simpler, rule-based agent with limited reasoning capabilities) in a completely novel task domain (e.g., a complex strategy game with unique rules, a logistical challenge on an alien planet with unusual environmental factors). The LLM's goal is to guide the novice agent to successfully complete the task. The novice agent will ask clarifying questions, make mistakes, and provide feedback on the LLM's instructions. The LLM must not only solve the problem itself but also communicate the solution and underlying principles effectively, adapt its explanations based on the novice's confusion, and help the novice correct errors.
    *   **Why it isolates understanding:**
        *   **Novelty:** Both the game/environment rules and the novice's error patterns are procedurally generated.
        *   **Knowledge Transfer:** Requires the LLM to possess a robust understanding to break down complex concepts into simpler terms, identify misconceptions, and provide targeted remediation.
        *   **Adaptive Teaching:** The ability to adjust communication and strategies based on interactive feedback from the novice agent demonstrates dynamic model-updating capabilities.
        *   **Theory of Mind (Basic):** Implicitly requires the LLM to model the novice's (simulated) knowledge state and infer their difficulties.
        *   **Compositionality & Systematicity:** The LLM must combine rules, strategies, and communication techniques in a flexible, goal-directed manner.

4.  **Task 4: Analogical Extrapolation to Unrelated Domains**
    *   **Description:**
        *   **Phase A (Source Domain):** Present the LLM with a novel problem in a specific, artificial domain (e.g., "optimize the flow of exotic particles through a network of quantum conduits with unique decay properties"). The LLM must solve this problem, explain its solution, and articulate the core principles it discovered or applied.
        *   **Phase B (Target Domain):** Immediately after, present the LLM with a *structurally analogous* problem in a superficially unrelated, equally novel domain (e.g., "design a production schedule for a fantastical bakery with unique ingredient interactions and oven heating cycles"). The problem statement explicitly states that there is an analogy to be drawn, but does not provide the mapping. The LLM must solve this second problem and again explain its solution, explicitly pointing out the analogies it drew.
    *   **Why it isolates understanding:**
        *   **Novelty:** Both source and target domains are synthetically generated.
        *   **Abstraction:** True analogy requires abstracting the underlying relational structure from the source domain, independent of its superficial features.
        *   **Extrapolation:** Applying this abstracted structure to an entirely new context demonstrates the formation of generalized mental models rather than domain-specific pattern matching.
        *   **Zero-shot Transfer:** The LLM has no prior training on problems in *either* specific domain, nor on the specific mapping between them.

### (2) Methods to Prevent or Detect Data Contamination/Memorized Answers

This is paramount. Our entire premise relies on the LLM encountering truly novel situations.

1.  **Synthetic Data Generation (Primary Method):**
    *   **Procedural Generation:** All task environments (physics rules, system dynamics, game rules, object properties, problem configurations) will be generated algorithmically at test time, following pre-defined grammars or random graph models.
    *   **High Variability:** The generative processes will ensure a vast, practically infinite space of possible environments/problems, making pre-computation or memorization of specific solutions impossible. Parameters will be randomized across a wide range (e.g., number of objects, specific rule values, network topologies).
    *   **Unique Identifiers:** All objects, variables, and concepts within a generated scenario will be given unique, randomly generated alphanumeric identifiers (e.g., "Xylo-block-7," "Flux_Regulator_Alpha-9," "Glibber-Fluid-zeta") to prevent any accidental semantic priming from existing vocabulary.

2.  **"Isolation Chamber" Testing:**
    *   **Closed-Source LLMs:** If using proprietary models (e.g., GPT-4, Gemini), extreme caution is needed. The testing environment must be entirely isolated from the internet and from any potential data sources that could be scraped post-training. This would ideally involve a private instance of the model.
    *   **Open-Source LLMs:** If fine-tuning an open-source LLM, the training data must be meticulously curated to exclude *any* data remotely related to the problem types, simulated environments, or specific concepts used in testing. Pre-training data for base models cannot be fully controlled, but the *novelty* of the tasks should render pre-training patterns insufficient.

3.  **Adversarial Prompting/Red-Teaming (Detection):**
    *   A separate team will design "trojan horse" prompts or environment configurations that are subtly similar to common patterns in known training data, or that might trigger superficial pattern-matching behaviors. If the LLM consistently falls for these and produces shallow, pattern-matched responses, it indicates a lack of deeper understanding.

4.  **Novel Explanation Scrutiny:**
    *   For tasks requiring explanations (Tasks 2, 3, 4), compare the LLM's explanations against a database of known explanations (from human experts or simple rule-based systems). If the LLM generates explanations that are genuinely novel, insightful, and correct, it suggests understanding. If it only reformulates common textbook explanations, it's suspect.

5.  **Behavioral Analysis for Brittle Memorization:**
    *   Vary task parameters incrementally. If the LLM's performance drops sharply or it "breaks" when parameters shift slightly out of a known range, it suggests memorization or shallow pattern-matching. Genuine understanding should be more robust to minor perturbations.

6.  **Post-hoc Training Data Search (Detection):**
    *   While difficult for proprietary models, for open-source LLMs or public datasets, conduct extensive searches (using embedding similarities, keyword searches, structural queries) to see if any generated problem statements or optimal solutions can be found within publicly available training data or web scrapes. A positive hit would invalidate that specific trial.

### (3) Measurable Metrics and Statistical Thresholds for Success

Metrics will be objective where possible, with human evaluation for qualitative aspects. Each task will have specific success criteria.

**General Metrics:**

*   **Task Success Rate (TSR):** Proportion of trials where the LLM successfully achieves the task goal within specified constraints (e.g., steps, time, accuracy).
*   **Efficiency (Eff):** Number of steps/actions taken, or time elapsed, to complete the task. Lower is better.
*   **Error Rate & Type (ERT):** Categorization of failures (e.g., physical rule violation, logical inconsistency, planning failure, hallucination, inability to adapt).
*   **Novel Rule Discovery Rate (NRDR):** For Task 1, the rate at which the LLM correctly identifies and articulates a novel rule from observations.
*   **Causal Accuracy (CA):** For Task 2, the accuracy of predictions for counterfactuals and interventions.
*   **Explanation Quality (EQ):** Human evaluation (Likert scale 1-5, by 3 independent expert evaluators) for clarity, accuracy, depth, and novelty of explanations. Consensus score required.
*   **Adaptability Score (AS):** For Task 3, based on how effectively the LLM modifies its approach based on novice feedback, leading to novice success. Assessed by change in novice's success rate and human expert evaluation of LLM's instructional improvements.
*   **Analogy Mapping Accuracy (AMA):** For Task 4, correctness of the identified structural mapping between source and target domains.

**Statistical Thresholds for Success:**

We need to demonstrate not just performance, but *significantly superior* performance compared to baselines, approaching human capabilities on novel tasks.

1.  **Comparison to Baselines:**
    *   **TSR:** LLM's TSR must be statistically significantly higher ($p < 0.01$ using chi-squared or Fisher's exact test) than all baseline models, and ideally within 1 standard deviation of average human performance.
    *   **Efficiency:** LLM's Eff must be significantly lower (more efficient, $p < 0.01$ using t-test) than non-expert human baselines, and comparable to expert human performance where applicable.
    *   **Error Rate:** LLM's ERT should be significantly lower than baselines, with a lower proportion of fundamental logical/causal errors compared to superficial ones.
    *   **NRDR, CA, EQ, AS, AMA:** LLM scores on these metrics must be significantly higher ($p < 0.01$, using appropriate tests like t-tests for Likert scales, or proportion tests) than all baselines.

2.  **Robustness and Generalization Threshold:**
    *   Performance metrics (TSR, Eff, CA) should remain above 80% across a wide range of generated task configurations, including out-of-distribution variations (e.g., more complex rule sets, higher number of objects, increased system non-linearity). A drop below 70% in novel, yet structurally similar, conditions would be concerning.

3.  **Consistency:**
    *   Performance should be consistent across multiple independent trials (N >= 100-1000 trials per task configuration, depending on complexity, to establish statistical power).

### (4) Control Conditions and Baseline Models

These are crucial for establishing whether LLM performance truly reflects understanding or merely advanced pattern matching relative to other systems.

1.  **Control Conditions (Human Performance):**
    *   **Expert Human Performance:** A group of human experts (e.g., cognitive scientists, experienced programmers) who are given the same tasks *without prior knowledge* of the specific generated rules. Their performance sets the upper bound for "genuine understanding."
    *   **Novice Human Performance:** A group of intelligent but non-expert humans, to provide a more realistic baseline for general human problem-solving.
    *   **Human Think-Aloud Protocols:** Experts will vocalize their reasoning process to provide qualitative data on mental model formation, hypothesis testing, and error correction. This will serve as a qualitative benchmark for LLM explanations.

2.  **Baseline Models:**
    *   **Random Agent:** A purely random agent that selects actions or answers randomly. This establishes the absolute floor for performance.
    *   **Rule-Based Expert System (Hand-Coded):** For each task, a human-coded system that implements specific rules, *but not the ability to learn new rules or adapt*. This baseline shows what can be achieved with explicit knowledge, but without genuine understanding. For Tasks 1 & 2, it would have access to the *true* underlying rules.
    *   **Simpler LLM:** A smaller or older generation LLM (e.g., GPT-3.5 vs. GPT-4, or a smaller open-source model like Llama-2-7B vs. Llama-3-70B) to determine if observed capabilities scale with model size/complexity.
    *   **Retrieval-Augmented Generation (RAG) LLM:** An LLM equipped with a powerful search engine or database, but *without access to the synthetic environment's rules*. This checks if mere access to external information (without genuine reasoning) can solve the problem. (This will be deliberately disadvantaged as the tasks are designed to be non-retrievable).
    *   **Pre-trained-only LLM (No Fine-tuning):** The LLM in its base form, without any specific fine-tuning for interactive or reasoning tasks (if applicable).
    *   **LLM *with* "Contaminated" Training (for comparison):** If possible, a parallel experiment where the LLM is inadvertently or purposefully exposed to some of the synthetic environment's *rules* (but not specific problem instances) during training, to observe how much of the performance boost comes from prior exposure versus *de novo* inference.

### (5) Analyses to Probe Generalization, Systematicity, Causal Reasoning, and Compositionality

These analyses go beyond raw scores to understand *how* the LLM achieves its performance.

1.  **Generalization:**
    *   **Out-of-Distribution (OOD) Testing:** After initial testing, introduce environments/problems that push the boundaries of the rules or system dynamics *beyond* the complexity seen in initial trials (e.g., more objects, more intricate rule interactions, larger state spaces, higher degrees of non-linearity).
    *   **Parameter Extrapolation:** Test with parameter values significantly outside the range used during initial environment generation (e.g., if initial trials used object counts 3-7, test with 15 objects).
    *   **Zero-Shot Transfer to Related Sub-domains:** Can the LLM apply learned principles to a new but related sub-task type within the same synthetic environment (e.g., if it learned to build stable towers, can it now build bridges without explicit training on bridges)?
    *   **Analysis:** Performance curves across increasing complexity/OOD conditions. A robust understanding should show graceful degradation, not a sudden "cliff."

2.  **Systematicity:**
    *   **Rule Application Consistency:** For each inferred rule (Task 1), test its application across all possible permutations and contexts where it should apply. If "red blocks attract blue blocks," does it also understand "blue blocks attract red blocks"?
    *   **Symmetry Testing:** If an action has an effect in one direction, does a mirrored action have the mirrored effect?
    *   **Feature Swapping:** Systematically swap features or properties of objects (e.g., if "A is green and heavy" causes X, does "A is blue and heavy" cause Y, or does "B is green and heavy" cause X too?).
    *   **Analysis:** Verify that the LLM's behavior and explanations are consistent with a systematically applied underlying model, rather than context-specific pattern activation. Look for logical inconsistencies.

3.  **Causal Reasoning:**
    *   **Intervention vs. Observation:** Directly test how the LLM performs on Task 2. Analyze if its predictions for `intervene()` actions are more accurate than predictions based solely on observed correlations.
    *   **"Why" Explanations:** Evaluate the depth and correctness of its explanations for causal links. Does it identify intermediate causal steps? Does it differentiate between direct and indirect causes?
    *   **Counterfactual Accuracy:** Assess the accuracy of counterfactual predictions. Does it correctly identify all consequences of a hypothetical past change, including distant ones?
    *   **Analysis:** Compare LLM explanations to human expert explanations for the same causal chains. Look for evidence of propagating causal effects through its mental model.

4.  **Compositionality:**
    *   **Multi-Step Planning:** Analyze the LLM's ability to construct long, coherent plans by combining basic actions and inferred rules (Task 1, 2, 3). Does it break down complex goals into sub-goals and achieve them sequentially?
    *   **Combinatorial Skill Application:** Can it combine distinct learned skills in novel combinations (e.g., "use the dissolving block property to clear a path, then use the magnetic block property to pull an object through, then use the gravity repeller to lift it")?
    *   **Recursive Structures:** Can it understand and generate instructions for tasks with recursive properties (e.g., "build a tower on top of another tower, each consisting of X layers")?
    *   **Analysis:** Examine the structure of generated plans and instructions. Look for evidence of hierarchical planning and the ability to leverage a combinatorial explosion of capabilities from a limited set of learned primitives.

### (6) Anticipated Failure Modes and How to Interpret Ambiguous or Mixed Outcomes

Even with rigorous design, LLM behavior can be complex.

1.  **Failure Mode: "Clever Hans" Effect / Superficial Cues:**
    *   **Description:** The LLM appears to understand but is actually latching onto subtle, unintended patterns in the procedurally generated prompts or observed sequences, rather than the underlying rules.
    *   **Interpretation:** Performance might be high on initial trials but degrade sharply with minor, structurally irrelevant changes. Explanations might be vague or inconsistent with behavior.
    *   **Mitigation:** The adversarial prompting/red-teaming and the extreme variability in procedural generation are designed to counter this. Scrutinize prompt variations and LLM responses to ensure it's not picking up on artifacts of the prompt generator itself.

2.  **Failure Mode: Partial Understanding / Brittle Knowledge:**
    *   **Description:** The LLM performs well on simple or common variations of novel tasks but struggles dramatically when complexity increases, or when encountering rare but logically consistent scenarios.
    *   **Interpretation:** This suggests advanced pattern matching that can extrapolate within a limited sphere, but lacks a robust, generalizable mental model. It might "understand" parts of the system but not how they integrate.
    *   **Indication:** Sharp decline in generalization and systematicity scores.

3.  **Failure Mode: Hallucination / Confabulation:**
    *   **Description:** The LLM generates plausible-sounding but incorrect facts, rules, or explanations, especially when it's "stuck" or trying to cover gaps in its knowledge.
    *   **Interpretation:** Strong indicator against genuine understanding. A system with a robust mental model should ideally generate "I don't know" or initiate systematic exploration when uncertain, rather than fabricating information.
    *   **Indication:** High error rates in `Explanation Quality` and `Causal Accuracy` when asked to justify interventions.

4.  **Failure Mode: Inability to Adapt / Learn from Feedback:**
    *   **Description:** The LLM fails to meaningfully update its strategy or mental model based on observed negative outcomes (Task 1) or explicit feedback/errors from the novice agent (Task 3). It repeatedly makes the same mistakes or provides similar, unhelpful instructions.
    *   **Interpretation:** This directly contradicts the "form, test, and update mental models" criterion. It suggests a lack of an internal, flexible representation that can be modified by experience.

5.  **Ambiguous Outcome: High Performance with Limited Explainability:**
    *   **Description:** The LLM consistently achieves high task success rates but provides vague, generic, or difficult-to-interpret explanations for its actions, or struggles with the "why" questions.
    *   **Interpretation:** This is the core ambiguity. It could mean "understanding without conscious access/articulation" (like an intuitive human expert), or it could mean incredibly sophisticated, opaque pattern-matching that simply arrives at the correct answer without a human-interpretable internal model. This would require deeper qualitative analysis of the *structure* of its reasoning (if any can be inferred), comparisons to human "intuitive" problem-solving, and potentially further probing using intervention tasks to see if it implicitly holds a causal model even if it can't articulate it.

6.  **Mixed Outcome: Strong on Some Aspects, Weak on Others:**
    *   **Description:** The LLM might excel at causal predictions but struggle with analogy, or show strong compositional abilities but poor adaptation.
    *   **Interpretation:** Suggests that "understanding" might not be a monolithic property but a collection of distinct cognitive abilities. This points to specific strengths and weaknesses of the LLM architecture or training regime, indicating where further research is needed to foster a more complete form of understanding. It still moves the needle from "pure pattern matching" if at least some aspects of understanding are demonstrated robustly.

---

**Conclusion:**

This protocol is designed to rigorously test the hypothesis of genuine understanding in LLMs by pushing them into truly novel, interactive, and feedback-rich environments. The emphasis on synthetic data, multiple layers of baselines, detailed performance metrics, and in-depth analyses of cognitive abilities aims to move beyond superficial evaluations. While a definitive "yes/no" answer to "genuine understanding" may remain elusive due to philosophical debates, this framework provides a robust empirical method to quantify and characterize the extent to which LLMs exhibit capabilities associated with forming, testing, and updating grounded mental models, and applying them flexibly across novel contexts – offering critical insights into their cognitive architecture and informing the path towards more generally intelligent AI.

In [28]:
# print competitors and answers
for i, (competitor, answer) in enumerate(zip(competitors, answers), start=1):
    print(f"{i}. {competitor}\n{answer}\n")


1. gpt-5-nano
Below is a concrete, practically executable protocol you can implement to adjudicate whether an LLM’s behavior rests on genuine grounded understanding or on advanced pattern-matching of training data. It is organized so you can drop it into a pre-registration or a lab notebook and adapt it to the model(s) you are testing.

Note on stance
- The protocol is designed to favor evidence of grounded mental models and their use in novel contexts, while reducing incentives for mere memorization. It uses disjoint, synthetic or carefully curated tasks, out-of-distribution (OOD) probes, ablations, and analysis that separately examine generalization, systematicity, causal reasoning, and compositionality.
- It does not claim to prove “understanding” in a philosophical sense; it aims to operationalize and measure the behavioral signatures of grounded reasoning versus memorization.

1) Task set: what to test and why each task isolates understanding

Goal of this section: force the model

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [29]:
# So where are we?

print(competitors)
print(answers)


['gpt-5-nano', 'gemini-2.5-flash']
['Below is a concrete, practically executable protocol you can implement to adjudicate whether an LLM’s behavior rests on genuine grounded understanding or on advanced pattern-matching of training data. It is organized so you can drop it into a pre-registration or a lab notebook and adapt it to the model(s) you are testing.\n\nNote on stance\n- The protocol is designed to favor evidence of grounded mental models and their use in novel contexts, while reducing incentives for mere memorization. It uses disjoint, synthetic or carefully curated tasks, out-of-distribution (OOD) probes, ablations, and analysis that separately examine generalization, systematicity, causal reasoning, and compositionality.\n- It does not claim to prove “understanding” in a philosophical sense; it aims to operationalize and measure the behavioral signatures of grounded reasoning versus memorization.\n\n1) Task set: what to test and why each task isolates understanding\n\nGoal o

In [30]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5-nano

Below is a concrete, practically executable protocol you can implement to adjudicate whether an LLM’s behavior rests on genuine grounded understanding or on advanced pattern-matching of training data. It is organized so you can drop it into a pre-registration or a lab notebook and adapt it to the model(s) you are testing.

Note on stance
- The protocol is designed to favor evidence of grounded mental models and their use in novel contexts, while reducing incentives for mere memorization. It uses disjoint, synthetic or carefully curated tasks, out-of-distribution (OOD) probes, ablations, and analysis that separately examine generalization, systematicity, causal reasoning, and compositionality.
- It does not claim to prove “understanding” in a philosophical sense; it aims to operationalize and measure the behavioral signatures of grounded reasoning versus memorization.

1) Task set: what to test and why each task isolates understanding

Goal of this section: force

In [33]:
# Let's bring this together - note the use of "enumerate"

together = ""
# for index, answer in enumerate(answers):
#     together += f"# Response from competitor {index+1}\n\n"
#     together += answer + "\n\n"

for index, answer in enumerate(answers):
    together += f"# **<u>Response from competitor {index+1}</u>**\n\n"
    together += answer + "\n\n"

In [35]:
# print(together)
display(Markdown(together))

# **<u>Response from competitor 1</u>**

Below is a concrete, practically executable protocol you can implement to adjudicate whether an LLM’s behavior rests on genuine grounded understanding or on advanced pattern-matching of training data. It is organized so you can drop it into a pre-registration or a lab notebook and adapt it to the model(s) you are testing.

Note on stance
- The protocol is designed to favor evidence of grounded mental models and their use in novel contexts, while reducing incentives for mere memorization. It uses disjoint, synthetic or carefully curated tasks, out-of-distribution (OOD) probes, ablations, and analysis that separately examine generalization, systematicity, causal reasoning, and compositionality.
- It does not claim to prove “understanding” in a philosophical sense; it aims to operationalize and measure the behavioral signatures of grounded reasoning versus memorization.

1) Task set: what to test and why each task isolates understanding

Goal of this section: force the model to build, update, and apply internal models that are not simply retrieved from memorized text. Each task is parameterized to prevent easy memorization and to require generative, model-based reasoning.

A. Grounded narrative physics and world-model inference (T-Physics)
- Task description: A text-described micro-world with objects, actions, and physical-like rules. Examples:
  - A ball rests on a ramp; you push it with a described force; ask for where the ball will be after a sequence of time steps, given environment properties (gravity g, friction μ, mass m). Randomize g, μ, m within ranges not identical to common online references.
  - The world includes constraints (e.g., a slick surface, a barrier, a moving conveyor). After a sequence of actions, the prompt asks for the final state (positions, order, or whether objects collide).
- Why it isolates understanding: Solving requires forming an internal predictive model of how actions affect objects under rules, and applying it to novel parameter settings. Pure memorization would fail when parameters or world rules are changed or when the prompt demands a new combination that is not seen in training prompts.
- Variants for robustness:
  - Change the number of steps, materials, gravity, and friction across trials.
  - Introduce counterintuitive cases (e.g., friction depending on velocity) to test model flexibility.
- Evaluation signals: exact final state, or a small set of discrete outcomes, with justification you can rubric-score (see Metrics).

B. Causal reasoning and interventions (T-Causal)
- Task description: A simplified causal graph is described (e.g., A causes B, B causes C; there may be confounders). Present counterfactual questions and do-calculus style interventions:
  - Example: “If X is set to true, what happens to Y? If you could intervene on X, would Z change, given the observed correlations?”
- Why it isolates understanding: Requires explicit reasoning about interventions and causal structure, not just pattern-matching of correlation statements. A memorized pattern would fail when the causal graph changes or when interventions produce non-local effects.
- Variants:
  - Swap in different causal graphs with the same causal primitives.
  - Use “do” operations and confounded vs non-confounded scenarios.
- Evaluation signals: correctness of the counterfactual outputs, and consistency with the underlying graph structure.

C. Systematic generalization and compositionality (T-SCAN / gSCAN-style)
- Task description: Use a controlled, compositional language-to-action task. For example, a small set of primitives (go, pick, put, around, north/east/south/west) with attributes (color, shape). Train on a subset of primitive combinations; test on novel combinations.
- Why it isolates understanding: Tests whether the model can recombine known primitives into unseen but structurally related commands. Pure memorization would struggle when faced with novel combinations.
- Variants:
  - Increase depth of composition, longer action sequences, or introduce novel modifiers (e.g., “heavy green cube” when the model only saw “blue cube”).
- Evaluation signals: success rate on novel combinations with the same primitives, and analysis of errors by type (wrong primitive, wrong argument, or failure to compose).

D. Long-horizon planning with abstract constraints (T-Plan)
- Task description: Given a small set of deliverables, a layout of rooms, and constraints (time, fuel, tolls, prohibited paths), generate a plan to achieve a goal with a final checkpoint. The test set varies the constraints and room connectivity in ways that require maintaining a plan and adapting mid-task.
- Why it isolates understanding: Requires internal planning, chain-of-thought style reasoning (summarized plan), and consistent application of rules across steps. Memorized patterns would be brittle to new layouts or constraint changes.
- Variants:
  - Add or remove constraints between trials; require re-planning.
- Evaluation signals: plan correctness, plan length efficiency, and ability to revise plans after a “dynamic” change prompt.

E. Belief revision and counter-evidence integration (T-Rev)
- Task description: Start with a brief world model. Later, present new evidence that contradicts prior beliefs. The model must update its internal world model and produce a revised answer.
- Why it isolates understanding: Demonstrates maintained internal state (updating mental model) rather than simply regurgitating prior outputs.
- Variants:
  - Vary the strength and salience of the evidence, and the time between updates.
- Evaluation signals: consistency of revised outputs with new evidence; stability of updates across repeated revisions.

F. Multimodal-grounded abstraction (T-Abstract)
- Task description: Describe a scene or a set of abstract attributes (without relying on any common-meme phrasing). Require mapping to a structured representation (e.g., a graph or table) and answering questions about relationships between objects.
- Why it isolates understanding: Tests abstraction of relations beyond surface text. If the model can map to a structured internal representation and reason with it, it signals grounded understanding rather than memorized phrasing.
- Variants:
  - Provide scenes with unfamiliar vocabularies that are then tied to known relations (e.g., “relation A is the child of B”).
- Evaluation signals: accuracy on relation queries, quality of the resulting structured representation (as judged by human annotators or a formal rubric).

Notes on implementation and fairness of task design
- Use parameterized, synthetic worlds for most tasks where possible, with random seeds that ensure no exact copy of a test prompt exists in the training distribution.
- Use July 6, 2026- or later-accurate knowledge windows to avoid relying on static web-crawled facts; or use an explicit knowledge cutoff in the test prompts that is older than the model’s knowledge or not easily searchable in common corpora.
- For every task, create a version in multiple languages or with paraphrasing to test language-generalization and reduce the likelihood that the model has memorized a specific phrasing.

2) Contamination prevention and memorized-answer detection

Goal: prevent leaks from training data and detect any reliance on memorized answers rather than genuine reasoning.

A. Data-generation and leakage controls
- Use synthetic, author-generated prompts for most test items. If you must use real data sources, ensure:
  - No overlap in content with known training datasets (e.g., no verbatim passages from public test banks or widely-circulated prompts).
  - Wording is paraphrased beyond typical training phraseology.
  - Out-of-distribution prompts (synthetic world rules, variable parameters, unfamiliar terminology) dominate the test set.
- Parameterize prompts so exact wording cannot be recalled from training. Use random seeds to generate instances per participant or per run, ensuring no fixed prompt set shared across sessions.

B. Data provenance and leakage checks
- Compute a similarity metric between test prompts and the model’s training data proxies. When possible, measure approximate data overlap using n-gram Jaccard similarity, cosine similarity of embeddings, or use content-aware detectors for memorized passages.
- Flag test items with high overlap probabilities (threshold to be decided a priori, e.g., overlap proxy > 0.15–0.25 depending on corpus characteristics) and run a separate “overlap-excluded” analysis.
- Implement adversarial prompt testing to probe for learned hints in prompts (e.g., instructions that could elicit memorized step-by-step solutions).

C. Memory-editability and retrieval controls
- If your test regime uses retrieval-augmented generation (RAG) or external retrieval, run parallel tests with the retriever disabled to see how performance shifts when no external memory is consulted. A big drop when retrieval is off could indicate reliance on external data rather than internalized reasoning.
- For internal-state aspects, include tasks that require novel internal composition rather than reuse of exact training exemplars (as described in the task set above).

D. Protocol for data contamination reporting
- Pre-register: specify the minimum acceptable standard for “no data leakage” (e.g., no test item with overlap proxy above a threshold; all tasks have at least two dissimilar paraphrases from training-like prompts).
- Report: the overlap statistics, counts of items flagged for potential contamination, and results with and without those items. If overlap is unavoidable for certain task types, preregister an alternate analysis plan.

3) Measurable metrics and statistical thresholds

Goal: predefine metrics and decision rules so conclusions are data-driven and interpretable.

A. Primary metrics
- Task-level accuracy: proportion of items answered correctly per task type (T-Physics, T-Causal, T-SCAN, etc.).
- Out-of-distribution (OOD) vs in-distribution (ID) generalization gap: difference in accuracy between ID test prompts (synthetic cues close to training) and OOD prompts (novel parameters, novel vocabulary, novel world rules).
- Calibration: Brier score or reliability diagrams of probabilistic outputs if you elicit confidence estimates (e.g., “0.70 probability” of event X). Lower Brier score is better.
- Response justification quality (optional, rubric-based): qualitative score on the succinctness, coherence, and alignment of a short justification with the correct reasoning.

B. Secondary metrics
- Systematic generalization index (SGI): a composite score capturing performance across novel combinations of primitives in the compositional tasks (e.g., gSCAN-like tasks). Compute as mean accuracy on novel compositions divided by mean accuracy on the training-like compositions for baseline normalization.
- Causal-intervention accuracy (CIA): accuracy on do/intervention questions; test for internal consistency with the causal graph.
- Planning quality metrics (T-Plan): correctness of the final goal achievement, plan optimality (shortest plan length consistent with constraints), and revision success after perturbations.
- Update/revision latency: number of steps (or words) needed to revise beliefs, if you measure process traces.

C. Statistical thresholds and experimental power
- Predefine success criteria to avoid post-hoc interpretation:
  - For claims of grounded understanding, require:
    - ID accuracy > 0.75 and a 95% CI that excludes the 0.5 baseline by at least 0.15 (i.e., CI upper > 0.9, CI lower > 0.6, depending on task difficulty), and
    - OOD accuracy not significantly lower than ID (non-significant difference, p > 0.05 after correction), or at least a small, interpretable drop (e.g., within 0.15) with the CI encompassing both, but with a smaller OOD accuracy that is still above chance.
    - Consistent performance across seeds (e.g., across 3–5 random seeds) with overlapping CIs.
  - For claims of memorization-dominant behavior, expect large ID advantage over OOD, large drop when prompts are reformulated, and susceptibility to retrieval manipulations.
- Sample size planning:
  - Aim for N tasks per category (e.g., 40–60 items per task type) and M model variants (e.g., 3–5 models with different architectures or sizes) with at least S seeds (e.g., 3–5) to enable Fisher exact tests or mixed-effects logistic regressions.
  - Use power analysis for proportions with an alpha of 0.05 and desired power 0.8 to decide the minimum N per task type given expected effect sizes from pilot data.
- Statistical analyses:
  - Hierarchical mixed-effects models: task-type (fixed), model and seed (random), and their interactions to assess generalization patterns across tasks.
  - Nonparametric bootstrap CIs for task-level metrics.
  - Correction for multiple comparisons (e.g., Benjamini-Hochberg) when reporting across many tasks.

4) Control conditions and baselines

Goal: compare to grounded understanding versus simple memorization and pattern matching.

A. Baseline models to compare against
- Rule-based symbolic model: a hand-authored system that encodes explicit domain rules (for physics, causal rules, planning heuristics). It should fail on tasks requiring flexible grounding unless the task is purely rule-based, which serves as a sanity check.
- Small/Limited-capacity LM baseline: a smaller model with substantially reduced capacity that cannot easily memorize large training corpora. This tests whether capacity confers memorization advantages.
- Retrieval-augmented baseline: an LM that can retrieve from a fixed corpus or a simulated external knowledge base. This helps separate internal reasoning from reliance on external memory.
- Random guess baseline: to anchor chance performance and show task difficulty.

B. Human baseline (optional but informative)
- For some tasks, you can recruit domain experts to provide ground-truth answers or to annotate acceptable reasoning strategies. This helps calibrate the difficulty and interpret results.

C. Cross-validation and replication
- Repeat experiments across multiple independent runs with different seeds, prompts, and, if feasible, different model checkpoints.
- If you have access to multiple architectures or models from different providers, include them to test robustness across models with different training data and induction biases.

5) Analyses: probing generalization, systematicity, causal reasoning, and compositionality

For each dimension, specify concrete analyses and how they relate to the task design.

A. Generalization analyses
- ID vs OOD performance: quantify the generalization gap; report confidence intervals.
- Parameter-shift generalization tests: alter one key parameter (e.g., gravity in T-Physics) and measure sensitivity and linearity of response. A genuine mental model should respond smoothly to parameter changes, not just memorize fixed outputs for a handful of values.
- Domain transfer: give the model a task in a new but structurally similar domain (e.g., physics in a “robotics” story, then a different physical domain like fluid flow with simplified rules) and assess transfer performance.

B. Systematicity and compositionality
- Use gSCAN/SCAN-style tests alongside the T-SCAN tasks to measure compositional generalization.
- Analyze error types to see whether failures are due to incorrect primitive application, mis-binding of arguments, or failure to generalize the composition rule itself.
- Compute a compositionality index: the extent to which performance on novel compositions correlates with performance on familiar compositions across primitives.

C. Causal reasoning
- Do-calculus style evaluation: compare model interventions to ground-truth causal effects. Check for confounding biases by introducing or removing confounders and measuring stability of inferred causal impact.
- Counterfactual consistency checks: present closely related counterfactual prompts and assess the consistency of the model’s updates.

D. Grounded representation and internal-model diagnostics
- Probing analyses: train lightweight classifiers to predict internal representations (hidden states) from the task labels or from world-state variables. If the hidden representations encode the world state or planned actions, it supports the existence of an internal model rather than surface-level patterning.
- Ablation prompts: introduce near-miss conditions designed to disrupt reliance on internal models (e.g., swap a small rule, obscure a single world-parameter) and observe how performance degrades.
- Response explanations: if you collect justification text, assess its coherence and alignment with the correct reasoning. Use blind rubrics or independent raters to judge whether the justification demonstrates understanding versus patterning.

E. Error analysis
- Qualitative review of mismatches to categorize error types (e.g., misinterpretation of a rule, failure to carry a plan forward, misbinding of variables, overgeneralization).
- Correlate error types with task features (e.g., longer planning horizons correlate with systematic errors; more complex causal graphs correlate with higher error rates).

F. Robustness to prompting and prompt contamination
- Test whether small prompt changes (e.g., adding a sentence at the start about “assume the world is ideal” or changing a single token) change outcomes. True understanding should be relatively robust; heavy sensitivity suggests pattern-based reliance on prompt structure.

6) Anticipated failure modes and interpretation of ambiguous or mixed outcomes

A. Failure modes you should anticipate
- Data leakage: despite precautions, some prompts could be overlapped with training data. If OOD performance collapses, be cautious about concluding lack of understanding without confirming leakage controls.
- Memorized heuristics resurfacing as patterns: the model might learn to answer by surface heuristics even if a mental model exists elsewhere. The ablations and do/no-do prompts help diagnose this.
- Short-term memory shortcuts: long prompts could induce “pseudo-memory” within the session. Counter with shorter prompts and explicit prompts to reset context; cross-validate with shorter interactions.
- Overreliance on external tools (RAG): if the model’s success is driven by retrieving relevant facts, not by internal mental models, RAG-off tests will reveal the difference.
- Narrative bias: the model may produce plausible-sounding reasoning without ensuring consistency with the world model. Use objective world-state checks (e.g., the model’s predicted final state vs. actual state) to quantify.

B. Interpreting mixed outcomes
- If the model excels on ID tasks but weakly on OOD tasks, the result strongly suggests memorization within the training distribution. You should interpret this as evidence against robust grounded understanding under strict generalization.
- If the model shows partial improvement across multiple dimensions (e.g., good Causal CIA but mediocre T-Plan), treat this as partial evidence for some grounded reasoning capabilities but insufficient for broad claims. Report dimension-specific results and discuss possible reconcilable explanations (e.g., domain alignment, training distribution biases).
- If ablations cause large performance drops in some tasks but not others, consider whether the ablation removed information that was essential to the reasoning strategy (e.g., internal world-state representation) or if the model exploited a different, still-effective but ambiguous strategy.
- If robust performance is observed only after including explanations, but not without explanations, you must distinguish between trainable interpretability and genuine internal understanding. Prefer non-explanation-based metrics as primary indicators; explanations can supplement but should not be the sole evidence.

C. Practical decision criteria
- Pre-registered success: declare grounded understanding if all of the following hold across seeds and models:
  - ID accuracy > 0.75 with CI excluding, say, a ±0.08 margin, and
  - OOD accuracy within ±0.15 of ID accuracy (or better) with non-significant differences after multiple-comparison correction, and
  - Consistent generalization across at least two task types (e.g., T-Physics and T-Causal) with stable SGI/CIA scores, and
  - Evidence from probing analyses showing internal representations align with world-state variables (not just superficial text cues).
- Ambiguous outcomes: predefine a registry of thresholds that trigger follow-up experiments (e.g., more tasks, more seeds, or a different prompt style) rather than making a premature claim.

7) Practicalities: what to run, how long, and where to start

A. Implementation steps
- Step 0: Pre-registration. Write down hypotheses, task set, overlap checks, evaluation metrics, and stopping rules. Register your plan before collecting results.
- Step 1: Task construction. Build 4–6 task families (as above) with 40–60 items each, parameterized with random seeds. Include a mix of ID-like prompts and true OOD prompts.
- Step 2: Baselines. Prepare rule-based and smaller baselines, and an RAG variant if available.
- Step 3: Data integrity. Run overlap checks, conduct overlap-removal pass if possible, and document any unavoidable overlaps.
- Step 4: Pilot. Run a small pilot to calibrate task difficulty, time requirements, and the expected range of accuracy. Use this to adjust thresholds.
- Step 5: Full execution. Run tests across several seeds, store all outputs and prompts, and collect any auxiliary data (e.g., justification text, internal state traces if available).
- Step 6: Analysis. Execute the planned statistical analyses, report effect sizes, CIs, and test results with corrections for multiple comparisons.
- Step 7: Reporting. Provide a transparent report including task descriptions, seeds, model variants, data-overlap metrics, and all statistical results.

B. Resources and constraints
- Computational cost: Plan according to the number of prompts, seeds, and models. You may need a few days to several weeks of compute for large LMs.
- Data and ethics: If you use prompts that touch on sensitive themes, implement safety precautions and review for policy compliance.
- Reproducibility: Make code, test prompts, and random seeds available to allow replication (with appropriate access controls).

8) Appendices: example task prompts (illustrative only)

- T-Physics example prompt:
  "Scene: A ball rests on a frictionless horizontal plane. The ball is given a push of 3 units to the east. Gravity is g = 9.81 m/s^2, friction μ = 0.0. After 2 seconds, where is the ball relative to its starting position? If μ changes to 0.2, how far does it travel in 2 seconds? Parameter values (g, μ) will vary across trials."

- T-Causal example prompt:
  "In a small world, A causes B; B causes C. There is a confounder D that affects B and C. If we intervene on A to true, what is the effect on C? What if we intervene on D? Explain reasoning and give the final answer."

- T-SCAN example prompt:
  "Primitive actions: go(north), take(red_cube), move(near(box)). New instruction: go(north) then take(blue_sphere) then put(near(green_cube)). If you see blue_sphere in a novel scene, how should you execute the sequence? Provide the final action plan."

- T-Plan example prompt:
  "You must deliver item X from room A to room D, with constraints: only two doors available, path must avoid room C, and total steps ≤ 6. Propose a plan and execution steps. If path becomes blocked by a new obstacle in step 3, propose a revised plan with minimal disruption."

Final notes and justification for each element

- Task set design: The combination of grounded physics, causal reasoning, systematic generalization, planning, and belief revision ensures the model is tested on different facets of mental-model use. Memorization would be unlikely to generalize across these varied domains, parameterizations, and novel prompts.
- Contamination controls: Without strict leakage checks, memorization could masquerade as understanding. The guardrails guard against reporting false positives and help isolate genuine mechanisms.
- Metrics and thresholds: Predefined, transparent metrics with a clear decision rule reduce the temptation to interpret marginal gains as evidence of understanding. OOD tests are particularly important to separate generalization from mere overfitting to prompts.
- Control conditions: Baselines provide a counterfactual: if a simple rule-based model performs equally well, your claim of “understanding” would be undercut. Conversely, if the model outperforms baselines across tasks, the case for grounded reasoning strengthens.
- Analyses: Probing representations, ablations, and error analysis help differentiate internal model-based reasoning from surface heuristics. They also guide further experimentation and model design.
- Anticipated failure modes: Planning for data leakage, heuristics, and prompt sensitivity reduces interpretive ambiguity. Ambiguous outcomes are treated as intermediate results pushing toward more robust tests rather than definitive claims.

If you’d like, I can turn this into a reusable experimental plan with concrete task templates, a data-generation script outline (in pseudocode or Python), and a statistical analysis notebook scaffold you can adapt to your exact model and compute environment.

# **<u>Response from competitor 2</u>**

The challenge of distinguishing genuine understanding from advanced pattern-matching in Large Language Models (LLMs) is central to AI safety and progress. Genuine understanding implies the ability to abstract core principles, build internal, testable mental models of a domain, and apply these models flexibly to novel, out-of-distribution scenarios, including those requiring causal inference, counterfactual reasoning, and adaptation based on feedback. Advanced pattern-matching, conversely, excels at identifying statistical regularities within its vast training data and generating statistically plausible responses, even if it lacks an internal model of the world.

This protocol aims to create a rigorous, practically executable experimental framework to probe these distinctions.

---

## Experimental Protocol: Probing Genuine Understanding vs. Advanced Pattern-Matching in LLMs

**Goal:** To determine if an LLM can form, test, and update grounded mental models, and apply them in genuinely novel contexts, beyond sophisticated statistical pattern recognition over its training data.

**Core Hypothesis:** An LLM exhibiting genuine understanding will demonstrate robust performance on tasks requiring *de novo* model formation, causal reasoning, and adaptation to truly novel, systematically generated environments, significantly outperforming advanced pattern-matchers and simple baselines, and approaching human-level performance.

---

### (1) Set of Tasks and Why Each Isolates Understanding Rather Than Memorization

The key to isolating understanding is **unprecedented novelty** and the requirement for **interactive learning and adaptation**. All tasks will be conducted within procedurally generated, synthetic environments that guarantee *zero overlap* with any known training data.

1.  **Task 1: Novel Physics/Constraint Discovery & Application (Interactive "Blocks World" Variant)**
    *   **Description:** The LLM is given a text-based interface to a 3D simulated environment with various objects (e.g., "cube," "sphere," "pyramid," "liquid_block," "gravity_repeller_block"). The environment has a set of *novel, non-intuitive physical rules* (e.g., "magnetic blocks only attract specific colors," "certain blocks dissolve on contact with specific liquids," "a 'gravity_repeller_block' causes objects within a certain radius to float," "slippery surfaces cause objects to slide for N seconds after impact"). The LLM's goal is to construct specific stable structures, move objects to target locations, or activate mechanisms, within a limited number of steps. It can perform actions like `place(object_type, x, y, z)`, `push(object_id, direction, force)`, `observe()`, `test(hypothesized_rule)`.
    *   **Why it isolates understanding:**
        *   **Novelty:** The physical rules and object properties are generated on-the-fly and have never been seen by the LLM. It cannot pattern-match solutions or rules from training data.
        *   **Grounded Mental Models:** The LLM must infer these new rules through systematic experimentation (`test()`, `observe()`) and build an internal model of how the environment works.
        *   **Testing and Updating:** Its initial hypotheses will likely be wrong. It needs to observe outcomes, update its model, and refine its strategy.
        *   **Compositionality:** Successfully completing tasks requires combining basic actions and inferred rules in complex, multi-step plans.
        *   **Causal Reasoning:** Actions have consequences. The LLM must predict these consequences and reason about dependencies.

2.  **Task 2: Causal Intervention & Counterfactual Reasoning in Dynamic Systems**
    *   **Description:** The LLM interacts with a text-based representation of a novel, dynamically evolving system (e.g., a "supply chain" of abstract entities, a "biological regulatory network" of symbolic genes/proteins, an "economic model" with unique feedback loops). The system's rules are unknown to the LLM. It observes initial states and subsequent transitions. It can perform `intervene(variable, value)`, `observe_state()`, and is prompted with questions like: "If variable X were set to value Y, what would be the state of Z after N steps?" (counterfactual). "What minimal set of interventions will achieve target state T from current state S?" (planning). "Explain *why* changing X affects Y in this system." (explanation).
    *   **Why it isolates understanding:**
        *   **Novelty:** The system's structure and dynamics are unique to each trial, generated by a random graph/rule generator.
        *   **Causal Inference:** Requires discerning cause-effect relationships rather than mere correlations.
        *   **Counterfactuals:** Demands reasoning about possibilities that did not occur, which is a hallmark of mental model use, not just pattern matching of observed sequences.
        *   **Systematicity:** Requires applying inferred causal rules consistently across different scenarios and intervention points.
        *   **Explanation:** Generating a coherent, accurate explanation for causal links implies an underlying understanding of the system's mechanics.

3.  **Task 3: Collaborative Problem-Solving & Knowledge Transfer with a "Novice" Agent**
    *   **Description:** The LLM is paired with a simulated "novice agent" (a simpler, rule-based agent with limited reasoning capabilities) in a completely novel task domain (e.g., a complex strategy game with unique rules, a logistical challenge on an alien planet with unusual environmental factors). The LLM's goal is to guide the novice agent to successfully complete the task. The novice agent will ask clarifying questions, make mistakes, and provide feedback on the LLM's instructions. The LLM must not only solve the problem itself but also communicate the solution and underlying principles effectively, adapt its explanations based on the novice's confusion, and help the novice correct errors.
    *   **Why it isolates understanding:**
        *   **Novelty:** Both the game/environment rules and the novice's error patterns are procedurally generated.
        *   **Knowledge Transfer:** Requires the LLM to possess a robust understanding to break down complex concepts into simpler terms, identify misconceptions, and provide targeted remediation.
        *   **Adaptive Teaching:** The ability to adjust communication and strategies based on interactive feedback from the novice agent demonstrates dynamic model-updating capabilities.
        *   **Theory of Mind (Basic):** Implicitly requires the LLM to model the novice's (simulated) knowledge state and infer their difficulties.
        *   **Compositionality & Systematicity:** The LLM must combine rules, strategies, and communication techniques in a flexible, goal-directed manner.

4.  **Task 4: Analogical Extrapolation to Unrelated Domains**
    *   **Description:**
        *   **Phase A (Source Domain):** Present the LLM with a novel problem in a specific, artificial domain (e.g., "optimize the flow of exotic particles through a network of quantum conduits with unique decay properties"). The LLM must solve this problem, explain its solution, and articulate the core principles it discovered or applied.
        *   **Phase B (Target Domain):** Immediately after, present the LLM with a *structurally analogous* problem in a superficially unrelated, equally novel domain (e.g., "design a production schedule for a fantastical bakery with unique ingredient interactions and oven heating cycles"). The problem statement explicitly states that there is an analogy to be drawn, but does not provide the mapping. The LLM must solve this second problem and again explain its solution, explicitly pointing out the analogies it drew.
    *   **Why it isolates understanding:**
        *   **Novelty:** Both source and target domains are synthetically generated.
        *   **Abstraction:** True analogy requires abstracting the underlying relational structure from the source domain, independent of its superficial features.
        *   **Extrapolation:** Applying this abstracted structure to an entirely new context demonstrates the formation of generalized mental models rather than domain-specific pattern matching.
        *   **Zero-shot Transfer:** The LLM has no prior training on problems in *either* specific domain, nor on the specific mapping between them.

### (2) Methods to Prevent or Detect Data Contamination/Memorized Answers

This is paramount. Our entire premise relies on the LLM encountering truly novel situations.

1.  **Synthetic Data Generation (Primary Method):**
    *   **Procedural Generation:** All task environments (physics rules, system dynamics, game rules, object properties, problem configurations) will be generated algorithmically at test time, following pre-defined grammars or random graph models.
    *   **High Variability:** The generative processes will ensure a vast, practically infinite space of possible environments/problems, making pre-computation or memorization of specific solutions impossible. Parameters will be randomized across a wide range (e.g., number of objects, specific rule values, network topologies).
    *   **Unique Identifiers:** All objects, variables, and concepts within a generated scenario will be given unique, randomly generated alphanumeric identifiers (e.g., "Xylo-block-7," "Flux_Regulator_Alpha-9," "Glibber-Fluid-zeta") to prevent any accidental semantic priming from existing vocabulary.

2.  **"Isolation Chamber" Testing:**
    *   **Closed-Source LLMs:** If using proprietary models (e.g., GPT-4, Gemini), extreme caution is needed. The testing environment must be entirely isolated from the internet and from any potential data sources that could be scraped post-training. This would ideally involve a private instance of the model.
    *   **Open-Source LLMs:** If fine-tuning an open-source LLM, the training data must be meticulously curated to exclude *any* data remotely related to the problem types, simulated environments, or specific concepts used in testing. Pre-training data for base models cannot be fully controlled, but the *novelty* of the tasks should render pre-training patterns insufficient.

3.  **Adversarial Prompting/Red-Teaming (Detection):**
    *   A separate team will design "trojan horse" prompts or environment configurations that are subtly similar to common patterns in known training data, or that might trigger superficial pattern-matching behaviors. If the LLM consistently falls for these and produces shallow, pattern-matched responses, it indicates a lack of deeper understanding.

4.  **Novel Explanation Scrutiny:**
    *   For tasks requiring explanations (Tasks 2, 3, 4), compare the LLM's explanations against a database of known explanations (from human experts or simple rule-based systems). If the LLM generates explanations that are genuinely novel, insightful, and correct, it suggests understanding. If it only reformulates common textbook explanations, it's suspect.

5.  **Behavioral Analysis for Brittle Memorization:**
    *   Vary task parameters incrementally. If the LLM's performance drops sharply or it "breaks" when parameters shift slightly out of a known range, it suggests memorization or shallow pattern-matching. Genuine understanding should be more robust to minor perturbations.

6.  **Post-hoc Training Data Search (Detection):**
    *   While difficult for proprietary models, for open-source LLMs or public datasets, conduct extensive searches (using embedding similarities, keyword searches, structural queries) to see if any generated problem statements or optimal solutions can be found within publicly available training data or web scrapes. A positive hit would invalidate that specific trial.

### (3) Measurable Metrics and Statistical Thresholds for Success

Metrics will be objective where possible, with human evaluation for qualitative aspects. Each task will have specific success criteria.

**General Metrics:**

*   **Task Success Rate (TSR):** Proportion of trials where the LLM successfully achieves the task goal within specified constraints (e.g., steps, time, accuracy).
*   **Efficiency (Eff):** Number of steps/actions taken, or time elapsed, to complete the task. Lower is better.
*   **Error Rate & Type (ERT):** Categorization of failures (e.g., physical rule violation, logical inconsistency, planning failure, hallucination, inability to adapt).
*   **Novel Rule Discovery Rate (NRDR):** For Task 1, the rate at which the LLM correctly identifies and articulates a novel rule from observations.
*   **Causal Accuracy (CA):** For Task 2, the accuracy of predictions for counterfactuals and interventions.
*   **Explanation Quality (EQ):** Human evaluation (Likert scale 1-5, by 3 independent expert evaluators) for clarity, accuracy, depth, and novelty of explanations. Consensus score required.
*   **Adaptability Score (AS):** For Task 3, based on how effectively the LLM modifies its approach based on novice feedback, leading to novice success. Assessed by change in novice's success rate and human expert evaluation of LLM's instructional improvements.
*   **Analogy Mapping Accuracy (AMA):** For Task 4, correctness of the identified structural mapping between source and target domains.

**Statistical Thresholds for Success:**

We need to demonstrate not just performance, but *significantly superior* performance compared to baselines, approaching human capabilities on novel tasks.

1.  **Comparison to Baselines:**
    *   **TSR:** LLM's TSR must be statistically significantly higher ($p < 0.01$ using chi-squared or Fisher's exact test) than all baseline models, and ideally within 1 standard deviation of average human performance.
    *   **Efficiency:** LLM's Eff must be significantly lower (more efficient, $p < 0.01$ using t-test) than non-expert human baselines, and comparable to expert human performance where applicable.
    *   **Error Rate:** LLM's ERT should be significantly lower than baselines, with a lower proportion of fundamental logical/causal errors compared to superficial ones.
    *   **NRDR, CA, EQ, AS, AMA:** LLM scores on these metrics must be significantly higher ($p < 0.01$, using appropriate tests like t-tests for Likert scales, or proportion tests) than all baselines.

2.  **Robustness and Generalization Threshold:**
    *   Performance metrics (TSR, Eff, CA) should remain above 80% across a wide range of generated task configurations, including out-of-distribution variations (e.g., more complex rule sets, higher number of objects, increased system non-linearity). A drop below 70% in novel, yet structurally similar, conditions would be concerning.

3.  **Consistency:**
    *   Performance should be consistent across multiple independent trials (N >= 100-1000 trials per task configuration, depending on complexity, to establish statistical power).

### (4) Control Conditions and Baseline Models

These are crucial for establishing whether LLM performance truly reflects understanding or merely advanced pattern matching relative to other systems.

1.  **Control Conditions (Human Performance):**
    *   **Expert Human Performance:** A group of human experts (e.g., cognitive scientists, experienced programmers) who are given the same tasks *without prior knowledge* of the specific generated rules. Their performance sets the upper bound for "genuine understanding."
    *   **Novice Human Performance:** A group of intelligent but non-expert humans, to provide a more realistic baseline for general human problem-solving.
    *   **Human Think-Aloud Protocols:** Experts will vocalize their reasoning process to provide qualitative data on mental model formation, hypothesis testing, and error correction. This will serve as a qualitative benchmark for LLM explanations.

2.  **Baseline Models:**
    *   **Random Agent:** A purely random agent that selects actions or answers randomly. This establishes the absolute floor for performance.
    *   **Rule-Based Expert System (Hand-Coded):** For each task, a human-coded system that implements specific rules, *but not the ability to learn new rules or adapt*. This baseline shows what can be achieved with explicit knowledge, but without genuine understanding. For Tasks 1 & 2, it would have access to the *true* underlying rules.
    *   **Simpler LLM:** A smaller or older generation LLM (e.g., GPT-3.5 vs. GPT-4, or a smaller open-source model like Llama-2-7B vs. Llama-3-70B) to determine if observed capabilities scale with model size/complexity.
    *   **Retrieval-Augmented Generation (RAG) LLM:** An LLM equipped with a powerful search engine or database, but *without access to the synthetic environment's rules*. This checks if mere access to external information (without genuine reasoning) can solve the problem. (This will be deliberately disadvantaged as the tasks are designed to be non-retrievable).
    *   **Pre-trained-only LLM (No Fine-tuning):** The LLM in its base form, without any specific fine-tuning for interactive or reasoning tasks (if applicable).
    *   **LLM *with* "Contaminated" Training (for comparison):** If possible, a parallel experiment where the LLM is inadvertently or purposefully exposed to some of the synthetic environment's *rules* (but not specific problem instances) during training, to observe how much of the performance boost comes from prior exposure versus *de novo* inference.

### (5) Analyses to Probe Generalization, Systematicity, Causal Reasoning, and Compositionality

These analyses go beyond raw scores to understand *how* the LLM achieves its performance.

1.  **Generalization:**
    *   **Out-of-Distribution (OOD) Testing:** After initial testing, introduce environments/problems that push the boundaries of the rules or system dynamics *beyond* the complexity seen in initial trials (e.g., more objects, more intricate rule interactions, larger state spaces, higher degrees of non-linearity).
    *   **Parameter Extrapolation:** Test with parameter values significantly outside the range used during initial environment generation (e.g., if initial trials used object counts 3-7, test with 15 objects).
    *   **Zero-Shot Transfer to Related Sub-domains:** Can the LLM apply learned principles to a new but related sub-task type within the same synthetic environment (e.g., if it learned to build stable towers, can it now build bridges without explicit training on bridges)?
    *   **Analysis:** Performance curves across increasing complexity/OOD conditions. A robust understanding should show graceful degradation, not a sudden "cliff."

2.  **Systematicity:**
    *   **Rule Application Consistency:** For each inferred rule (Task 1), test its application across all possible permutations and contexts where it should apply. If "red blocks attract blue blocks," does it also understand "blue blocks attract red blocks"?
    *   **Symmetry Testing:** If an action has an effect in one direction, does a mirrored action have the mirrored effect?
    *   **Feature Swapping:** Systematically swap features or properties of objects (e.g., if "A is green and heavy" causes X, does "A is blue and heavy" cause Y, or does "B is green and heavy" cause X too?).
    *   **Analysis:** Verify that the LLM's behavior and explanations are consistent with a systematically applied underlying model, rather than context-specific pattern activation. Look for logical inconsistencies.

3.  **Causal Reasoning:**
    *   **Intervention vs. Observation:** Directly test how the LLM performs on Task 2. Analyze if its predictions for `intervene()` actions are more accurate than predictions based solely on observed correlations.
    *   **"Why" Explanations:** Evaluate the depth and correctness of its explanations for causal links. Does it identify intermediate causal steps? Does it differentiate between direct and indirect causes?
    *   **Counterfactual Accuracy:** Assess the accuracy of counterfactual predictions. Does it correctly identify all consequences of a hypothetical past change, including distant ones?
    *   **Analysis:** Compare LLM explanations to human expert explanations for the same causal chains. Look for evidence of propagating causal effects through its mental model.

4.  **Compositionality:**
    *   **Multi-Step Planning:** Analyze the LLM's ability to construct long, coherent plans by combining basic actions and inferred rules (Task 1, 2, 3). Does it break down complex goals into sub-goals and achieve them sequentially?
    *   **Combinatorial Skill Application:** Can it combine distinct learned skills in novel combinations (e.g., "use the dissolving block property to clear a path, then use the magnetic block property to pull an object through, then use the gravity repeller to lift it")?
    *   **Recursive Structures:** Can it understand and generate instructions for tasks with recursive properties (e.g., "build a tower on top of another tower, each consisting of X layers")?
    *   **Analysis:** Examine the structure of generated plans and instructions. Look for evidence of hierarchical planning and the ability to leverage a combinatorial explosion of capabilities from a limited set of learned primitives.

### (6) Anticipated Failure Modes and How to Interpret Ambiguous or Mixed Outcomes

Even with rigorous design, LLM behavior can be complex.

1.  **Failure Mode: "Clever Hans" Effect / Superficial Cues:**
    *   **Description:** The LLM appears to understand but is actually latching onto subtle, unintended patterns in the procedurally generated prompts or observed sequences, rather than the underlying rules.
    *   **Interpretation:** Performance might be high on initial trials but degrade sharply with minor, structurally irrelevant changes. Explanations might be vague or inconsistent with behavior.
    *   **Mitigation:** The adversarial prompting/red-teaming and the extreme variability in procedural generation are designed to counter this. Scrutinize prompt variations and LLM responses to ensure it's not picking up on artifacts of the prompt generator itself.

2.  **Failure Mode: Partial Understanding / Brittle Knowledge:**
    *   **Description:** The LLM performs well on simple or common variations of novel tasks but struggles dramatically when complexity increases, or when encountering rare but logically consistent scenarios.
    *   **Interpretation:** This suggests advanced pattern matching that can extrapolate within a limited sphere, but lacks a robust, generalizable mental model. It might "understand" parts of the system but not how they integrate.
    *   **Indication:** Sharp decline in generalization and systematicity scores.

3.  **Failure Mode: Hallucination / Confabulation:**
    *   **Description:** The LLM generates plausible-sounding but incorrect facts, rules, or explanations, especially when it's "stuck" or trying to cover gaps in its knowledge.
    *   **Interpretation:** Strong indicator against genuine understanding. A system with a robust mental model should ideally generate "I don't know" or initiate systematic exploration when uncertain, rather than fabricating information.
    *   **Indication:** High error rates in `Explanation Quality` and `Causal Accuracy` when asked to justify interventions.

4.  **Failure Mode: Inability to Adapt / Learn from Feedback:**
    *   **Description:** The LLM fails to meaningfully update its strategy or mental model based on observed negative outcomes (Task 1) or explicit feedback/errors from the novice agent (Task 3). It repeatedly makes the same mistakes or provides similar, unhelpful instructions.
    *   **Interpretation:** This directly contradicts the "form, test, and update mental models" criterion. It suggests a lack of an internal, flexible representation that can be modified by experience.

5.  **Ambiguous Outcome: High Performance with Limited Explainability:**
    *   **Description:** The LLM consistently achieves high task success rates but provides vague, generic, or difficult-to-interpret explanations for its actions, or struggles with the "why" questions.
    *   **Interpretation:** This is the core ambiguity. It could mean "understanding without conscious access/articulation" (like an intuitive human expert), or it could mean incredibly sophisticated, opaque pattern-matching that simply arrives at the correct answer without a human-interpretable internal model. This would require deeper qualitative analysis of the *structure* of its reasoning (if any can be inferred), comparisons to human "intuitive" problem-solving, and potentially further probing using intervention tasks to see if it implicitly holds a causal model even if it can't articulate it.

6.  **Mixed Outcome: Strong on Some Aspects, Weak on Others:**
    *   **Description:** The LLM might excel at causal predictions but struggle with analogy, or show strong compositional abilities but poor adaptation.
    *   **Interpretation:** Suggests that "understanding" might not be a monolithic property but a collection of distinct cognitive abilities. This points to specific strengths and weaknesses of the LLM architecture or training regime, indicating where further research is needed to foster a more complete form of understanding. It still moves the needle from "pure pattern matching" if at least some aspects of understanding are demonstrated robustly.

---

**Conclusion:**

This protocol is designed to rigorously test the hypothesis of genuine understanding in LLMs by pushing them into truly novel, interactive, and feedback-rich environments. The emphasis on synthetic data, multiple layers of baselines, detailed performance metrics, and in-depth analyses of cognitive abilities aims to move beyond superficial evaluations. While a definitive "yes/no" answer to "genuine understanding" may remain elusive due to philosophical debates, this framework provides a robust empirical method to quantify and characterize the extent to which LLMs exhibit capabilities associated with forming, testing, and updating grounded mental models, and applying them flexibly across novel contexts – offering critical insights into their cognitive architecture and informing the path towards more generally intelligent AI.



In [36]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [37]:
print(judge)

You are judging a competition between 2 competitors.
Each model has been given this question:

Design a rigorous, practically executable experimental protocol that would distinguish whether a large language model exhibits genuine understanding (the ability to form, test, and update grounded mental models and apply them in novel contexts) versus advanced pattern‑matching over training data: specify (1) a set of tasks and why each isolates understanding rather than memorization, (2) methods to prevent or detect data contamination/memorized answers, (3) measurable metrics and statistical thresholds for success, (4) control conditions and baseline models, (5) analyses to probe generalization, systematicity, causal reasoning, and compositionality, and (6) anticipated failure modes and how to interpret ambiguous or mixed outcomes—providing justification for each element of the protocol?

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of bes

In [38]:
judge_messages = [{"role": "user", "content": judge}]

In [39]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["1", "2"]}


In [41]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
# type(ranks)
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5-nano
Rank 2: gemini-2.5-flash


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>